In [0]:
#Load and inspect the members table
df_members = spark.table("default.members")
df_members.printSchema()
df_members.display()

In [0]:
#Determine if any employee is current or terminated

from pyspark.sql import functions as F

df_elig = df_members.withColumn(
    "coverage_status",
    F.when(F.col("termination_date").isNull(), "Current")
    .otherwise("Terminated")
)

# df_elig labels members as current if the termination date is null, otherwise terminated by adding an additional column labled "coverage_status" with two categories
# either "Current" or "Terminated".


#display info by coverage status
df_elig.groupBy("coverage_status").count().display()

df_coverage_summary = df_elig.groupBy("coverage_status").count()


In [0]:
# Check for data quality to see if current employee is missing critical registration info

df_dq = df_elig.withColumn(
    "missing_phone", F.col("phone").isNull()
).withColumn(
    "missing_email", F.col("email").isNull()
).withColumn(
    "missing_zip", F.col("zip_code").isNull()
).withColumn(
    "missing_dob", F.col("dob").isNull()
).withColumn(
    "missing_critical_info",
    F.col("missing_phone") | F.col("missing_email") | F.col("missing_zip") | F.col("missing_dob")
)
# This code checks to if any critical info is missing but first it create a column for each critical and a cumulative column for if nay conatct column is missing info

#df_dq.display()

df_dq_summary = df_dq

In [0]:

# Show only members with at least one missing critical field
df_dq.filter(F.col("missing_critical_info")).select(
    "member_id", "member_name", "coverage_status",
    "missing_phone", "missing_email", "missing_zip", "missing_dob"
).display()

# Quick summary: how many active members have incomplete registration data
df_dq.filter(F.col("missing_critical_info") & (F.col("coverage_status") == "Active")) \
    .count()


In [0]:
df_dq_summary = df_dq.filter(F.col("coverage_status") == "Current").agg(
    F.sum(F.col("missing_phone").cast("int")).alias("missing_phone_count"),
    F.sum(F.col("missing_email").cast("int")).alias("missing_email_count"),
    F.sum(F.col("missing_zip").cast("int")).alias("missing_zip_count"),
    F.sum(F.col("missing_dob").cast("int")).alias("missing_dob_count"),
    F.count("*").alias("total_current_members")
)


In [0]:
df_visits = spark.table("default.patient_visits")

df_check = df_visits.join(df_elig, "member_id") \
    .withColumn(
        "was_eligible_at_visit",
        F.when(
            (F.col("visit_date") >= F.col("enrollment_date")) &
            (F.col("termination_date").isNull() | (F.col("visit_date") <= F.col("termination_date"))),
            True
        ).otherwise(False)
    )
# The above code joins eligible patients table we created with patient visits table
# It verifies if a member was eligible at the time of their visit, based on their enrollment and termination dates.
# If the visit date is between the enrollment date and the termination date (or if there is no termination date), the member is considered eligible.
# Otherwise, the member is considered not eligible

df_check.groupBy("was_eligible_at_visit").count().display()

#was_eligible_at_visit count
#False               1375
#True                374

In [0]:
#Here we begin investigating different conditions for why patients were not eligble for claims

In [0]:
# To begin we will isolate ineglible patients at the time of visit
df_ineligible_visits = df_check.filter(F.col("was_eligible_at_visit") == False)
# we will extract only key columns

df_ineligible_visits.select(
    "visit_id", "member_id", "member_name", "visit_date",
    "enrollment_date", "termination_date", "coverage_status"
).display()

# Breakdown: how many are "visit before enrollment" vs "visit after termination"
df_ineligible_visits.withColumn(
    "ineligibility_reason",
    F.when(F.col("visit_date") < F.col("enrollment_date"), "Visit before enrollment")
     .when(F.col("visit_date") > F.col("termination_date"), "Visit after termination")
     .otherwise("Unknown")
).groupBy("ineligibility_reason").count().display()

In [0]:
# Now we tie this inelibile patient visits to the cliams table and see if these were correctly or erronously paid

In [0]:
df_claims = spark.table("default.claims")

df_claims.columns


In [0]:
# Since pyspark 
v = df_ineligible_visits.alias("v")
c = df_claims.alias("c")

df_leakage_check = v.join(c, v.visit_id == c.visit_id, "left") \
    .select(
        "v.visit_id", "v.member_id", "v.member_name", "v.visit_date",
        "c.claim_status", "c.denial_reason", "c.paid_amount"
    )
# We join the claims table to the ineligble visits table to see if any of these claims were paid
df_leakage_summary = df_leakage_check.groupBy("claim_status").agg(
    F.count("*").alias("claim_count"),
    F.sum(F.coalesce(F.col("paid_amount"), F.lit(0.0))).alias("total_paid_amount")
)

In [0]:
df_leakage_export = df_leakage_check.withColumn(
    "paid_amount", F.coalesce(F.col("paid_amount"), F.lit(0.0))
)

In [0]:
df_leakage_check.columns

In [0]:
df_leakage_check.display()

In [0]:
# We examine the reasons why cliams were denied
df_leakage_check.filter(
    (F.col("claim_status") == "Denied") & (F.col("denial_reason").isNotNull())
).groupBy("denial_reason").count().display()

In [0]:
# Denied claims missing a reason — a data quality finding in its own right
df_leakage_check.filter(
    (F.col("claim_status") == "Denied") & (F.col("denial_reason").isNull())
).select("visit_id", "member_id", "claim_status").display()

In [0]:
# we will examine the claims that were paid to inelibile visits
df_elig_paid = df_leakage_check.filter(F.col("claim_status") == "Paid")

In [0]:
df_elig_paid.filter(F.col("paid_amount").isNotNull()).agg(F.sum("paid_amount").alias("total_paid")).display()

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.exports")

In [0]:
df_coverage_summary.toPandas().to_csv(
    "/Volumes/workspace/default/hc_db/gold_coverage_status_summary.csv", index=False
)
df_dq_summary.toPandas().to_csv(
    "/Volumes/workspace/default/hc_db/gold_data_quality_summary.csv", index=False
)
df_leakage_export.toPandas().to_csv(
    "/Volumes/workspace/default/hc_db/gold_eligibility_leakage_detail.csv", index=False
)
df_leakage_summary.toPandas().to_csv(
    "/Volumes/workspace/default/hc_db/gold_eligibility_leakage_summary.csv", index=False
)

In [0]:
## Part 2: Claims Adjudication / Denial Analysis (denial rate, Days in A/R, net collection rate) ##



In [0]:

# with this code we can calculate the denial rate
# we will first calculate the total number of claims and the number of denied claims
# then we will calculate the percentage of denied claims by dividing the number of denied claims by the total number of claims
df_denial_rate= df_claims.agg(
        F.count("*").alias("total_claims"),
        F.sum(F.when(F.col("claim_status") == "Denied",1).otherwise(0)).alias("denied_claims")
).withColumn(
    "denial_rate_pct",
    F.round(F.col("denied_claims") / F.col("total_claims") * 100, 2)
)
df_denial_rate.display()

In [0]:
df_ar = df_claims.filter(F.col("claim_status") == "Paid") \
    .withColumn(
        "days_in_ar",
        F.datediff(F.col("claim_paid_date"), F.col("claim_date"))
    )

df_ar_summary = df_ar.agg(
    F.avg("days_in_ar").alias("avg_days_in_ar"),
    F.expr("percentile_approx(days_in_ar, 0.5)").alias("median_days_in_ar"),
    F.max("days_in_ar").alias("max_days_in_ar"),
    F.min("days_in_ar").alias("min_days_in_ar"),
)

df_ar_summary.display()

In [0]:
df_ncr = df_claims.agg(
    F.sum(F.coalesce(F.col('paid_amount'), F.lit(0.0))).alias("total_paid"),
    F.sum(F.col("allowed_amount")).alias("total_allowed")
).withColumn(
    "net_collection_rate_pct",
    F.round(F.col("total_paid") / F.col("total_allowed") * 100, 2)
)

df_ncr.display()

In [0]:
df_writeoff = df_claims.filter(F.col("claim_status") == "Paid") \
    .withColumn(
        "writeoff_amount",
        F.col("allowed_amount") - F.col("paid_amount")
    )

df_writeoff_summary = df_writeoff.agg(
    F.sum("writeoff_amount").alias("total_writeoff"),
    F.avg("writeoff_amount").alias("avg_writeoff_per_claim"),
    F.count(F.when(F.col("writeoff_amount") > 0, 1)).alias("claims_with_writeoff")
)

df_writeoff_summary.display()

In [0]:
# Denial rate
denial_metrics = df_claims.agg(
    F.count("*").alias("total_claims"),
    F.sum(F.when(F.col("claim_status") == "Denied", 1).otherwise(0)).alias("denied_claims")
).withColumn(
    "denial_rate_pct",
    F.round(F.col("denied_claims") / F.col("total_claims") * 100, 2)
)

# Days in A/R (Paid claims only)
ar_metrics = df_claims.filter(F.col("claim_status") == "Paid") \
    .withColumn("days_in_ar", F.datediff(F.col("claim_paid_date"), F.col("claim_date"))) \
    .agg(
        F.round(F.avg("days_in_ar"), 2).alias("avg_days_in_ar"),
        F.expr("percentile_approx(days_in_ar, 0.5)").alias("median_days_in_ar")
    )

# Net collection rate
ncr_metrics = df_claims.agg(
    F.sum(F.coalesce(F.col("paid_amount"), F.lit(0.0))).alias("total_paid"),
    F.sum(F.col("allowed_amount")).alias("total_allowed")
).withColumn(
    "net_collection_rate_pct",
    F.round(F.col("total_paid") / F.col("total_allowed") * 100, 2)
)

# Write-off variance (Paid claims only)
writeoff_metrics = df_claims.filter(F.col("claim_status") == "Paid") \
    .withColumn("writeoff_amount", F.col("allowed_amount") - F.col("paid_amount")) \
    .agg(
        F.round(F.sum("writeoff_amount"), 2).alias("total_writeoff"),
        F.round(F.avg("writeoff_amount"), 2).alias("avg_writeoff_per_claim"),
        F.count(F.when(F.col("writeoff_amount") > 0, 1)).alias("claims_with_writeoff")
    )

# Combine all four into one row using crossJoin (each is a single-row df)
df_denial_analysis_summary = denial_metrics.crossJoin(ar_metrics) \
    .crossJoin(ncr_metrics) \
    .crossJoin(writeoff_metrics) \
    .select(
        "total_claims",
        "denied_claims",
        "denial_rate_pct",
        "avg_days_in_ar",
        "median_days_in_ar",
        "total_paid",
        "total_allowed",
        "net_collection_rate_pct",
        "total_writeoff",
        "avg_writeoff_per_claim",
        "claims_with_writeoff"
    )

df_denial_analysis_summary.display()

In [0]:
df_denial_analysis_summary.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/hc_db/gold_denial_analysis_summary"
)

In [0]:
df_provider_denials = df_claims.groupBy("provider_id").agg(
    F.count("*").alias("total_claims"),
    F.sum(F.when(F.col("claim_status") == "Denied", 1).otherwise(0)).alias("denied_claims")
).withColumn(
    "denial_rate_pct",
    F.round(F.col("denied_claims") / F.col("total_claims") * 100, 2)
).orderBy(F.col("denial_rate_pct").desc())

df_provider_denials.display()

In [0]:
df_providers = spark.table("workspace.default.providers")

p = df_provider_denials.alias("p")
prov = df_providers.alias("prov")

df_provider_denials_named = p.join(prov, p.provider_id == prov.provider_id, "left") \
    .select(
        "p.provider_id",
        "prov.provider_name",
        "p.total_claims",
        "p.denied_claims",
        "p.denial_rate_pct"
    ) \
    .orderBy(F.col("denial_rate_pct").desc())

df_provider_denials_named.display()


In [0]:
spark.table("workspace.default.claim_procedures").printSchema()

In [0]:
df_proc = spark.table("workspace.default.claim_procedures")

cp = df_proc.alias("cp")
c = df_claims.alias("c")

df_procedure_denials = cp.join(c, cp.claim_id == c.claim_id, "left") \
    .select("cp.claim_id", "cp.cpt_code", "cp.procedure_name", "c.claim_status") \
    .dropDuplicates(["claim_id", "procedure_name"]) \
    .groupBy("cpt_code", "procedure_name") \
    .agg(
        F.count("*").alias("total_claims"),
        F.sum(F.when(F.col("claim_status") == "Denied", 1).otherwise(0)).alias("denied_claims")
    ) \
    .withColumn(
        "denial_rate_pct",
        F.round(F.col("denied_claims") / F.col("total_claims") * 100, 2)
    ) \
    .orderBy(F.col("denial_rate_pct").desc())

df_procedure_denials.display()

In [0]:
df_procedure_denials = cp.join(c, cp.claim_id == c.claim_id, "left") \
    .select("cp.claim_id", "cp.cpt_code", "cp.procedure_name", "c.claim_status") \
    .dropDuplicates(["claim_id", "cpt_code"]) \
    .groupBy("cpt_code") \
    .agg(
        F.first("procedure_name").alias("procedure_name"),
        F.count("*").alias("total_claims"),
        F.sum(F.when(F.col("claim_status") == "Denied", 1).otherwise(0)).alias("denied_claims")
    ) \
    .withColumn(
        "denial_rate_pct",
        F.round(F.col("denied_claims") / F.col("total_claims") * 100, 2)
    ) \
    .orderBy(F.col("denial_rate_pct").desc())

df_procedure_denials.display()

In [0]:
df_procedure_denials.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/hc_db/gold_procedure_denial_rates"
)

In [0]:
df_denial_reasons = df_claims.filter(
    (F.col("claim_status") == "Denied") & (F.col("denial_reason").isNotNull())
).groupBy("denial_reason").agg(
    F.count("*").alias("claim_count")
).orderBy(F.col("claim_count").desc())

df_denial_reasons.display()

In [0]:
df_monthly_trend = df_claims.withColumn(
    "claim_month", F.date_format(F.col("claim_date"), "yyyy-MM")
).groupBy("claim_month").agg(
    F.count("*").alias("total_claims"),
    F.sum(F.when(F.col("claim_status") == "Denied", 1).otherwise(0)).alias("denied_claims")
).withColumn(
    "denial_rate_pct", F.round(F.col("denied_claims") / F.col("total_claims") * 100, 2)
).orderBy("claim_month")

df_monthly_trend.display()

In [0]:
totals = df_claims.agg(
    F.sum("claimed_amount").alias("total_claimed"),
    F.sum("allowed_amount").alias("total_allowed"),
    F.sum(F.coalesce(F.col("paid_amount"), F.lit(0.0))).alias("total_paid")
).collect()[0]

df_funnel = spark.createDataFrame([
    ("Claimed", totals["total_claimed"]),
    ("Allowed", totals["total_allowed"]),
    ("Paid", totals["total_paid"])
], ["stage", "amount"])

df_funnel.display()

In [0]:
df_denial_reasons.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/hc_db/gold_denial_reasons"
)

df_monthly_trend.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/hc_db/gold_monthly_claims_trend"
)

df_funnel.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/hc_db/gold_claims_funnel"
)

In [0]:
df_claims.groupBy("claim_status").count().display()

In [0]:
df_leakage_check.groupBy("claim_status").count().display()

In [0]:
df_leakage_check_clean = df_leakage_check.withColumn(
    "claim_status",
    F.coalesce(F.col("claim_status"), F.lit("No Claim Filed"))
)

In [0]:
df_leakage_check_clean.groupBy("claim_status").count().display()

In [0]:
df_leakage_check_clean.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/hc_db/gold_eligibility_leakage_detail"
)